In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from climate_attitudes.settings import Config, RawDataFile
from climate_attitudes.visualisation import configure_mpl

configure_mpl(Path("../fonts"))

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(156)

config = Config(_env_file="../.env")

In [ ]:
data = (
    pl.read_parquet(RawDataFile.Waves1to5Responses.filepath(config))
    .filter(pl.col("PID").is_not_null())
    .with_columns(
        pl.col("WAVE").cast(int).alias("wave"),
        pl.col("StartDate", "EndDate").str.strptime(
            pl.Datetime, format="%-m/%-d/%y %R", strict=True
        ),
    )
    .rename({"PID": "participant_id"})
    .with_columns(pl.col("wave").min().over("participant_id").alias("wave_joined"))
    .with_columns(
        pl.when(pl.col("wave") == pl.col("wave_joined"))
        .then(pl.lit("new"))
        .otherwise(pl.lit("repeating"))
        .alias("participant_type")
    )
)

In [ ]:
data.filter(wave=1).select(pl.col("valid").value_counts())

In [ ]:
data.filter(pl.col("cc14").is_not_null()).select("wave").unique()

In Wave 1 several participants have no identifier. Sara advises these are likely respondents who had no intention of repeating. Here we check how many such participants there were.

In [ ]:
data = (
    pl.read_parquet(RawDataFile.Waves1to5Responses.filepath(config))
    .filter(pl.col("PID").is_not_null())
    .with_columns(
        pl.col("WAVE").cast(int).alias("wave"),
        pl.col("StartDate", "EndDate").str.strptime(
            pl.Datetime, format="%-m/%-d/%y %R", strict=True
        ),
    )
    .rename({"PID": "participant_id"})
    .with_columns(pl.col("wave").min().over("participant_id").alias("wave_joined"))
    .with_columns(
        pl.when(pl.col("wave") == pl.col("wave_joined"))
        .then(pl.lit("new"))
        .otherwise(pl.lit("repeating"))
        .alias("participant_type")
    )
)

w3_new = data.filter(wave=3, participant_type="new")
w3_repeating = data.filter(wave=3, participant_type="repeating")

w5_new = data.filter(wave=5, participant_type="new")
w5_repeating = data.filter(wave=5, participant_type="repeating")

In Wave 2 we observe 380 non-null responses to `ew1` from repeating participants, while this question is intended only for new participants. 

Possible that this is a Qualtrics error, where repeating participants were shown the wrong survey initially, and this was later corrected. We can check this by examining the survey start times of the erroneous and non-erroneous responses.

In [ ]:
w2_repeating = data.filter(wave=2, participant_type="repeating")

In [ ]:
err_pids = (
    w2_repeating.select("participant_id", "ew1")
    .filter(
        ~pl.col("ew1").is_not_null(),
    )
    .select(pl.col("participant_id").cast(int).sort())
)
err_pids

In [ ]:
fig, axes = plt.subplots(
    ncols=2, figsize=(7, 1.7), sharey=True, constrained_layout=True
)

err_pids = (
    w2_repeating.select("participant_id", "ew1")
    .filter(
        pl.col("ew1").is_not_null(),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

plot_data = w2_repeating.select("StartDate", "EndDate", "participant_id").with_columns(
    pl.when(pl.col("participant_id").is_in(err_pids.to_series().implode()))
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("err")
)

ok_responses = [start_date for start_date, *_, err in plot_data.iter_rows() if not err]
err_responses = [start_date for start_date, *_, err in plot_data.iter_rows() if err]
axes[0].eventplot(
    [ok_responses, err_responses],
    orientation="horizontal",
    lineoffsets=[0, 1],
    linewidth=0.05,
    linelengths=0.7,
    colors=["b", "r"],
)

axes[0].set_yticks([0, 1], labels=["Ok", "Error"])
axes[0].set_title("Wave 2: Repeating, responds to 'only-new'")

err_pids = (
    w2_repeating.select(
        "participant_id",
        "cc_pol_RE.research",
    )
    .filter(
        pl.col(
            "cc_pol_RE.research"
        ).is_null()  # | (pl.col("cc_pol_RE.research") == ""),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

plot_data = w2_repeating.select("StartDate", "EndDate", "participant_id").with_columns(
    pl.when(pl.col("participant_id").is_in(err_pids.to_series().implode()))
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("err")
)

ok_responses = [start_date for start_date, *_, err in plot_data.iter_rows() if not err]
err_responses = [start_date for start_date, *_, err in plot_data.iter_rows() if err]
axes[1].eventplot(
    [ok_responses, err_responses],
    orientation="horizontal",
    lineoffsets=[0, 1],
    linewidth=0.05,
    linelengths=0.7,
    colors=["b", "r"],
)

axes[1].set_title("Wave 2: Repeating don't respond to 'all-participants'")
axes[1].tick_params(axis="both", which="both", length=0)

fig.autofmt_xdate(rotation=45)
for ax in axes.flatten():
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# fig.savefig("w2_switchpoints.png", dpi=200, bbox_inches="tight");

In [ ]:
err_pids = (
    w3_repeating.select("participant_id", "ew1")
    .filter(
        pl.col("ew1").is_not_null(),
        pl.col("ew1") != "",
    )
    .select(pl.col("participant_id").cast(int).sort())
)

plot_data = w3_repeating.select("StartDate", "EndDate", "participant_id").with_columns(
    pl.when(pl.col("participant_id").is_in(err_pids.to_series().implode()))
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("err")
)


fig, ax = plt.subplots(figsize=(6, 1))
sns.scatterplot(
    plot_data.sort(by="StartDate"), x="StartDate", y="err", s=10, alpha=0.7, ax=ax
)
ax.set_yticks([0, 1], labels=["False", "True"])
ax.set_ylabel("Error")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate(rotation=45)
ax.set_title("Wave 3: Some repeating respond to 'only-new'");

In [ ]:
err_pids = (
    w3_new.select("participant_id", "ew1_jun")
    .filter(
        pl.col("ew1_jun").is_not_null(),
        pl.col("ew1_jun") != "",
    )
    .select(pl.col("participant_id").cast(int).sort())
)

plot_data = w3_new.select("StartDate", "EndDate", "participant_id").with_columns(
    pl.when(pl.col("participant_id").is_in(err_pids.to_series().implode()))
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("err")
)


fig, ax = plt.subplots(figsize=(6, 1))
sns.scatterplot(
    plot_data.sort(by="StartDate"), x="StartDate", y="err", s=10, alpha=0.7, ax=ax
)
ax.set_yticks([0, 1], labels=["False", "True"])
ax.set_ylabel("Error")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate(rotation=45)
ax.set_title("Wave 3: Some new respond to 'only-repeating'");

In [ ]:
fig, axes = plt.subplots(
    ncols=2, figsize=(7, 1.7), sharey=True, constrained_layout=True
)

err_pids = (
    w3_new.select("participant_id", "ew1_jun")
    .filter(
        pl.col("ew1_jun").is_not_null(),
        pl.col("ew1_jun") != "",
    )
    .select(pl.col("participant_id").cast(int).sort())
)

plot_data = w3_new.select("StartDate", "EndDate", "participant_id").with_columns(
    pl.when(pl.col("participant_id").is_in(err_pids.to_series().implode()))
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("err")
)

ok_responses = [start_date for start_date, *_, err in plot_data.iter_rows() if not err]
err_responses = [start_date for start_date, *_, err in plot_data.iter_rows() if err]
axes[0].eventplot(
    [ok_responses, err_responses],
    orientation="horizontal",
    lineoffsets=[0, 1],
    linewidth=0.05,
    linelengths=0.7,
    colors=["b", "r"],
)

axes[0].set_yticks([0, 1], labels=["Ok", "Error"])
axes[0].set_title("Wave 3: Some new respond to 'only-repeating'")

err_pids = (
    w5_new.select(
        "participant_id",
        "cc_impact_1",
    )
    .filter(
        pl.col("cc_impact_1").is_null()  # | (pl.col("ew1_jun") == ""),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

plot_data = w5_new.select("StartDate", "EndDate", "participant_id").with_columns(
    pl.when(pl.col("participant_id").is_in(err_pids.to_series().implode()))
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("err")
)

ok_responses = [start_date for start_date, *_, err in plot_data.iter_rows() if not err]
err_responses = [start_date for start_date, *_, err in plot_data.iter_rows() if err]
axes[1].eventplot(
    [ok_responses, err_responses],
    orientation="horizontal",
    lineoffsets=[0, 1],
    linewidth=0.05,
    linelengths=0.7,
    colors=["b", "r"],
)

axes[1].set_title("Wave 5: Some new don't respond to 'all-participants'")


axes[1].tick_params(axis="both", which="both", length=0)

fig.autofmt_xdate(rotation=45)
for ax in axes.flatten():
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# fig.savefig("w3and5_errors.png", dpi=200, bbox_inches="tight");

In [ ]:
print(type([ok_responses, err_responses]))

In [ ]:
type(ok_responses[0])

In [ ]:
err_pids = (
    w5_new.select(
        "participant_id",
        "cc_impact_1",
    )
    .filter(
        pl.col("cc_impact_1").is_null()  # | (pl.col("ew1_jun") == ""),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

plot_data = w5_new.select("StartDate", "EndDate", "participant_id").with_columns(
    pl.when(pl.col("participant_id").is_in(err_pids.to_series().implode()))
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("err")
)


fig, ax = plt.subplots(figsize=(6, 1))
sns.scatterplot(
    plot_data.sort(by="StartDate"), x="StartDate", y="err", s=10, alpha=0.7, ax=ax
)
ax.set_yticks([0, 1], labels=["False", "True"])
ax.set_ylabel("Error")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate(rotation=45)
ax.set_title("Wave 5: Some new don't respond to 'all-participants'");

In [ ]:
err_pids = (
    w5_repeating.select(
        "participant_id",
        "cc_impact_1",
    )
    .filter(
        pl.col("cc_impact_1").is_null()  # | (pl.col("ew1_jun") == ""),
    )
    .select(pl.col("participant_id").cast(int).sort())
)

plot_data = w5_repeating.select("StartDate", "EndDate", "participant_id").with_columns(
    pl.when(pl.col("participant_id").is_in(err_pids.to_series().implode()))
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("err")
)


fig, ax = plt.subplots(figsize=(6, 1))
sns.scatterplot(
    plot_data.sort(by="StartDate"), x="StartDate", y="err", s=10, alpha=0.7, ax=ax
)
ax.set_yticks([0, 1], labels=["False", "True"])
ax.set_ylabel("Error")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate(rotation=45)
ax.set_title("Wave 5: Some repeating don't respond to 'all-participants'");